# 🏥 Tech Challenge FIAP - Fase 1
## Notebook EXTRA: Visão Computacional com CNN (OPCIONAL)

---

### 📋 **Objetivos deste Notebook (EXTRA)**

1. ✅ Converter features para formato de imagem
2. ✅ Criar arquitetura CNN (Convolutional Neural Network)
3. ✅ Treinar rede neural profunda
4. ✅ Comparar com modelos clássicos
5. ✅ Visualizar aprendizado da CNN
6. ✅ Análise de overfitting/underfitting

---

**⚠️ ATENÇÃO: NOTEBOOK OPCIONAL**

Este notebook é **OPCIONAL** e serve para:
- Demonstrar habilidades avançadas em Deep Learning
- Explorar abordagem alternativa ao problema
- Ganhar crédito extra no Tech Challenge

**Não é obrigatório para completar o projeto!**

---

### **🤔 Por que CNN para dados tabulares?**

Normalmente CNNs são usadas para imagens. Mas podemos:
1. Converter features para "imagens" (reshape)
2. Aproveitar arquitetura convolucional para detectar padrões
3. Comparar performance com modelos clássicos

**Nota**: Para este dataset específico, modelos clássicos (RF, XGBoost)  
provavelmente terão performance igual ou melhor. CNN é mais um exercício  
de exploração de técnicas avançadas.

## 📚 1. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Deep Learning - TensorFlow/Keras
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
    from tensorflow.keras.utils import to_categorical
    print(f"✅ TensorFlow versão: {tf.__version__}")
    print(f"✅ Keras disponível")
except ImportError:
    print("⚠️ TensorFlow não instalado!")
    print("   Execute: pip install tensorflow")
    print("   Este notebook requer TensorFlow.")

# Métricas
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

# Utilitários
import pickle
import time

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
RANDOM_STATE = 42

# Seed para reprodutibilidade
np.random.seed(RANDOM_STATE)
if 'tf' in dir():
    tf.random.set_seed(RANDOM_STATE)

print("\n✅ Bibliotecas importadas com sucesso!")

## 📂 2. Carregamento dos Dados

In [ ]:
# Carregar dados pré-processados
print("📥 Carregando dados...")

X_train = np.load('../data/processed/X_train.npy')
X_test = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test = np.load('../data/processed/y_test.npy')

# Carregar nomes das features
with open('../data/processed/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

print("✅ Dados carregados")
print(f"\n📊 DIMENSÕES:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test: {y_test.shape}")
print(f"\n📋 Features: {len(feature_names)}")

## 🔄 3. Preparação dos Dados para CNN

In [ ]:
# Para CNN, precisamos reformatar os dados
# Features tabulares: (n_samples, n_features)
# Para CNN 1D: (n_samples, n_features, 1)
# Para CNN 2D: (n_samples, height, width, 1)

print("🔄 Preparando dados para CNN...\n")

# Opção 1: CNN 1D (mais adequada para dados tabulares)
X_train_cnn1d = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn1d = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("📊 CNN 1D:")
print(f"   X_train shape: {X_train_cnn1d.shape}")
print(f"   X_test shape: {X_test_cnn1d.shape}")

# Opção 2: CNN 2D (reshape para formato de imagem)
# 30 features -> 6x5 "imagem"
height, width = 6, 5
X_train_cnn2d = X_train.reshape(X_train.shape[0], height, width, 1)
X_test_cnn2d = X_test.reshape(X_test.shape[0], height, width, 1)

print("\n📊 CNN 2D:")
print(f"   X_train shape: {X_train_cnn2d.shape}")
print(f"   X_test shape: {X_test_cnn2d.shape}")
print(f"   'Imagem': {height}x{width} pixels")

print("\n✅ Dados preparados para ambas arquiteturas")

## 🏗️ 4. Arquitetura CNN 1D

In [ ]:
# Construir modelo CNN 1D
print("🏗️ Construindo CNN 1D...\n")

def create_cnn1d_model(input_shape):
    """
    CNN 1D para dados tabulares
    """
    model = models.Sequential([
        # Camada 1: Convolução
        layers.Conv1D(filters=64, kernel_size=3, activation='relu', 
                     input_shape=input_shape, padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.3),
        
        # Camada 2: Convolução
        layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.3),
        
        # Camada 3: Convolução
        layers.Conv1D(filters=256, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.4),
        
        # Camadas densas
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        
        # Saída (classificação binária)
        layers.Dense(1, activation='sigmoid')
    ])
    
    return model

# Criar modelo
cnn1d_model = create_cnn1d_model(input_shape=(X_train_cnn1d.shape[1], 1))

# Compilar
cnn1d_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print("✅ Modelo CNN 1D criado e compilado")
print("\n📊 ARQUITETURA:")
cnn1d_model.summary()

## 🏗️ 5. Arquitetura CNN 2D

In [ ]:
# Construir modelo CNN 2D
print("🏗️ Construindo CNN 2D...\n")

def create_cnn2d_model(input_shape):
    """
    CNN 2D para dados tabulares formatados como imagem
    """
    model = models.Sequential([
        # Camada 1: Convolução
        layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu',
                     input_shape=input_shape, padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.25),
        
        # Camada 2: Convolução
        layers.Conv2D(filters=64, kernel_size=(2, 2), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Dropout(0.25),
        
        # Flatten
        layers.Flatten(),
        
        # Camadas densas
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        
        # Saída
        layers.Dense(1, activation='sigmoid')
    ])
    
    return model

# Criar modelo
cnn2d_model = create_cnn2d_model(input_shape=(height, width, 1))

# Compilar
cnn2d_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print("✅ Modelo CNN 2D criado e compilado")
print("\n📊 ARQUITETURA:")
cnn2d_model.summary()

## 🏋️ 6. Treinamento - CNN 1D

In [ ]:
# Callbacks
callbacks_1d = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        '../models/cnn1d_best.h5',
        monitor='val_auc',
        save_best_only=True,
        mode='max',
        verbose=0
    )
]

print("🏋️ Treinando CNN 1D...")
print("   (Isso pode levar alguns minutos...)\n")

start_time = time.time()

history_1d = cnn1d_model.fit(
    X_train_cnn1d, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks_1d,
    verbose=1
)

training_time_1d = time.time() - start_time

print(f"\n✅ CNN 1D treinada em {training_time_1d:.2f} segundos ({training_time_1d/60:.2f} minutos)")

## 🏋️ 7. Treinamento - CNN 2D

In [ ]:
# Callbacks
callbacks_2d = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        '../models/cnn2d_best.h5',
        monitor='val_auc',
        save_best_only=True,
        mode='max',
        verbose=0
    )
]

print("🏋️ Treinando CNN 2D...")
print("   (Isso pode levar alguns minutos...)\n")

start_time = time.time()

history_2d = cnn2d_model.fit(
    X_train_cnn2d, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks_2d,
    verbose=1
)

training_time_2d = time.time() - start_time

print(f"\n✅ CNN 2D treinada em {training_time_2d:.2f} segundos ({training_time_2d/60:.2f} minutos)")

## 📊 8. Visualização do Treinamento

In [ ]:
# Plotar curvas de aprendizado - CNN 1D
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_1d.history['loss'], label='Treino', linewidth=2)
axes[0].plot(history_1d.history['val_loss'], label='Validação', linewidth=2)
axes[0].set_title('CNN 1D - Loss', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history_1d.history['accuracy'], label='Treino', linewidth=2)
axes[1].plot(history_1d.history['val_accuracy'], label='Validação', linewidth=2)
axes[1].set_title('CNN 1D - Accuracy', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/21_cnn1d_training.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico CNN 1D salvo em: reports/figures/21_cnn1d_training.png")

In [ ]:
# Plotar curvas de aprendizado - CNN 2D
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_2d.history['loss'], label='Treino', linewidth=2)
axes[0].plot(history_2d.history['val_loss'], label='Validação', linewidth=2)
axes[0].set_title('CNN 2D - Loss', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history_2d.history['accuracy'], label='Treino', linewidth=2)
axes[1].plot(history_2d.history['val_accuracy'], label='Validação', linewidth=2)
axes[1].set_title('CNN 2D - Accuracy', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/22_cnn2d_training.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico CNN 2D salvo em: reports/figures/22_cnn2d_training.png")

## 📊 9. Avaliação nos Dados de Teste

In [ ]:
# Avaliar CNN 1D
print("📊 Avaliando CNN 1D no conjunto de teste...\n")

loss_1d, acc_1d, auc_1d = cnn1d_model.evaluate(X_test_cnn1d, y_test, verbose=0)
y_pred_1d_proba = cnn1d_model.predict(X_test_cnn1d, verbose=0).flatten()
y_pred_1d = (y_pred_1d_proba > 0.5).astype(int)

print("🤖 CNN 1D - RESULTADOS NO TESTE:")
print(f"   Loss: {loss_1d:.4f}")
print(f"   Accuracy: {acc_1d:.4f} ({acc_1d*100:.2f}%)")
print(f"   AUC: {auc_1d:.4f} ({auc_1d*100:.2f}%)")

print("\n📋 Relatório de Classificação (CNN 1D):")
print(classification_report(y_test, y_pred_1d, 
                          target_names=['Maligno (0)', 'Benigno (1)']))

In [ ]:
# Avaliar CNN 2D
print("📊 Avaliando CNN 2D no conjunto de teste...\n")

loss_2d, acc_2d, auc_2d = cnn2d_model.evaluate(X_test_cnn2d, y_test, verbose=0)
y_pred_2d_proba = cnn2d_model.predict(X_test_cnn2d, verbose=0).flatten()
y_pred_2d = (y_pred_2d_proba > 0.5).astype(int)

print("🤖 CNN 2D - RESULTADOS NO TESTE:")
print(f"   Loss: {loss_2d:.4f}")
print(f"   Accuracy: {acc_2d:.4f} ({acc_2d*100:.2f}%)")
print(f"   AUC: {auc_2d:.4f} ({auc_2d*100:.2f}%)")

print("\n📋 Relatório de Classificação (CNN 2D):")
print(classification_report(y_test, y_pred_2d,
                          target_names=['Maligno (0)', 'Benigno (1)']))

## 📊 10. Comparação: CNN vs Modelos Clássicos

In [ ]:
# Carregar resultados dos modelos clássicos
try:
    classic_results = pd.read_csv('../reports/model_comparison.csv')
    
    # Adicionar resultados das CNNs
    cnn_results = pd.DataFrame([
        {
            'Model': 'CNN 1D',
            'Accuracy': acc_1d,
            'Precision': None,  # Calcular depois
            'Recall': None,
            'F1-Score': None,
            'ROC-AUC': auc_1d,
            'Training Time (s)': training_time_1d
        },
        {
            'Model': 'CNN 2D',
            'Accuracy': acc_2d,
            'Precision': None,
            'Recall': None,
            'F1-Score': None,
            'ROC-AUC': auc_2d,
            'Training Time (s)': training_time_2d
        }
    ])
    
    # Combinar
    all_results = pd.concat([classic_results, cnn_results], ignore_index=True)
    
    print("📊 COMPARAÇÃO: MODELOS CLÁSSICOS vs CNNs")
    print("="*100)
    print(all_results[['Model', 'Accuracy', 'ROC-AUC', 'Training Time (s)']].to_string(index=False))
    print("="*100)
    
    # Salvar
    all_results.to_csv('../reports/model_comparison_with_cnn.csv', index=False)
    print("\n✅ Comparação salva em: reports/model_comparison_with_cnn.csv")
    
except FileNotFoundError:
    print("⚠️ Arquivo de resultados clássicos não encontrado")
    print("   Execute o Notebook 03 primeiro.")

## 📊 11. Matrizes de Confusão - CNNs

In [ ]:
# Plotar matrizes de confusão
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# CNN 1D
cm_1d = confusion_matrix(y_test, y_pred_1d)
sns.heatmap(cm_1d, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           cbar=False, square=True, linewidths=2)
axes[0].set_title('CNN 1D - Matriz de Confusão', fontweight='bold')
axes[0].set_xlabel('Predição')
axes[0].set_ylabel('Real')
axes[0].set_xticklabels(['Maligno', 'Benigno'])
axes[0].set_yticklabels(['Maligno', 'Benigno'])

# CNN 2D
cm_2d = confusion_matrix(y_test, y_pred_2d)
sns.heatmap(cm_2d, annot=True, fmt='d', cmap='Greens', ax=axes[1],
           cbar=False, square=True, linewidths=2)
axes[1].set_title('CNN 2D - Matriz de Confusão', fontweight='bold')
axes[1].set_xlabel('Predição')
axes[1].set_ylabel('Real')
axes[1].set_xticklabels(['Maligno', 'Benigno'])
axes[1].set_yticklabels(['Maligno', 'Benigno'])

plt.tight_layout()
plt.savefig('../reports/figures/23_cnn_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/23_cnn_confusion_matrices.png")

## 📊 12. Resumo do Notebook Extra

In [ ]:
print("="*100)
print("📊 RESUMO - NOTEBOOK EXTRA (CNN)")
print("="*100)

print("\n✅ MODELOS TREINADOS:")
print("   1. CNN 1D (para dados tabulares)")
print("   2. CNN 2D (dados formatados como imagem)")

print("\n📊 RESULTADOS:")
print(f"   CNN 1D:")
print(f"      • Accuracy: {acc_1d:.4f} ({acc_1d*100:.2f}%)")
print(f"      • AUC: {auc_1d:.4f}")
print(f"      • Tempo: {training_time_1d:.2f}s")
print(f"\n   CNN 2D:")
print(f"      • Accuracy: {acc_2d:.4f} ({acc_2d*100:.2f}%)")
print(f"      • AUC: {auc_2d:.4f}")
print(f"      • Tempo: {training_time_2d:.2f}s")

print("\n💾 ARQUIVOS GERADOS:")
print("   • models/cnn1d_best.h5")
print("   • models/cnn2d_best.h5")
print("   • reports/model_comparison_with_cnn.csv")
print("   • reports/figures/21_cnn1d_training.png")
print("   • reports/figures/22_cnn2d_training.png")
print("   • reports/figures/23_cnn_confusion_matrices.png")

print("\n🎯 OBSERVAÇÕES:")
print("   • CNNs são mais complexas e demoram mais para treinar")
print("   • Para este dataset, modelos clássicos provavelmente têm performance similar")
print("   • CNNs são mais adequadas para dados com estrutura espacial (imagens reais)")
print("   • Este exercício demonstra versatilidade técnica")

print("\n" + "="*100)

## 📝 13. Conclusões do Notebook Extra

### **🔍 O que aprendemos:**

1. **CNNs para Dados Tabulares**:
   - É possível adaptar CNNs para dados não-imagem
   - CNN 1D trata features como sequência
   - CNN 2D trata features como "imagem" artificial

2. **Trade-offs**:
   - ✅ CNNs podem capturar padrões complexos
   - ✅ Arquitetura flexível
   - ❌ Mais lentas para treinar
   - ❌ Menos interpretáveis
   - ❌ Requerem mais dados para generalizar bem

3. **Quando usar CNNs**:
   - ✅ Dados com estrutura espacial (imagens, vídeos)
   - ✅ Sequências temporais (CNN 1D)
   - ✅ Datasets muito grandes
   - ❌ Dados tabulares pequenos (use Random Forest, XGBoost)

### **💡 Insights:**

Para o Wisconsin Breast Cancer Database:
- Modelos clássicos (RF, XGBoost) são mais adequados
- CNNs não oferecem vantagem significativa
- Dataset é pequeno (~569 amostras) e estruturado
- Features não têm relação espacial natural

### **🏆 Valor do Exercício:**

Mesmo sem superioridade de performance, este notebook demonstra:
1. Conhecimento de Deep Learning
2. Capacidade de adaptar técnicas
3. Pensamento crítico sobre escolha de modelos
4. Habilidade com TensorFlow/Keras

### **⚠️ Recomendação Final:**

> Para o Tech Challenge, **USE os modelos clássicos** (RF, XGBoost, SVM).  
> Eles são:  
> - Mais rápidos  
> - Mais interpretáveis (compatível com SHAP)  
> - Igualmente eficazes  
> - Mais adequados para dados tabulares  

As CNNs ficam como **demonstração de versatilidade técnica**!

---

## ✅ Conclusão do Notebook EXTRA

**Status**: ✅ Completo (OPCIONAL)

**Modelos Treinados**: 2 (CNN 1D + CNN 2D)

**Valor**: Crédito extra + Demonstração de habilidades avançadas

**Recomendação**: Use modelos clássicos para entrega principal